In [3]:
#  imports and config
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report
import json
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

PARQUET_PATH  = r"C:\Users\super\Desktop\Data Thesis\final_df_clustered.parquet"
BASELINE_PT   = r"C:\Users\super\Desktop\Data Thesis\BaselinesGrid\transformer_best.pt"
BEST_PARAMS   = r"C:\Users\super\Desktop\Data Thesis\BaselinesGrid\transformer_best_params.json"
MAX_SEQ_LEN   = 80
BATCH_SIZE    = 32
EPOCHS        = 50
PATIENCE      = 10
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# thresholds from baseline results
BASELINE_MACRO_F1        = 0.6142
BASELINE_WORST_GROUP_ERR = 0.3023
BASELINE_EOD             = 0.1508
MACRO_F1_FLOOR           = BASELINE_MACRO_F1 - 0.05   # 0.5642

print(f"Using device: {DEVICE}")
print(f"\nTargets for fair model:")
print(f"  Macro F1 ≥         {MACRO_F1_FLOOR:.4f}")
print(f"  Worst-group error < {BASELINE_WORST_GROUP_ERR:.4f}")
print(f"  EOD <               {BASELINE_EOD:.4f}")

Using device: cpu

Targets for fair model:
  Macro F1 ≥         0.5642
  Worst-group error < 0.3023
  EOD <               0.1508


In [4]:
# Load data, reconstruct split
from sklearn.preprocessing import StandardScaler

print("Loading data...")
final_df = pd.read_parquet(PARQUET_PATH)

gaze_cols = [c for c in final_df.columns if "gaze" in c]
pose_cols = [c for c in final_df.columns if "pose" in c]
au_r_cols = [c for c in final_df.columns if c.startswith("AU") and c.endswith("_r")]
au_c_cols = [c for c in final_df.columns if c.startswith("AU") and c.endswith("_c")]

FEATURE_COLS = au_r_cols + au_c_cols + gaze_cols + pose_cols
N_FEATURES   = len(FEATURE_COLS)

assert "behavioral_cluster" not in FEATURE_COLS

# Reconstruct the exact same split as the baseline notebook
participant_label = (
    final_df.groupby("participant_id")["engagement"]
    .agg(lambda x: int(x.mean() >= 0.5))
)
engaged_participants    = participant_label[participant_label == 1].index.values.copy()
disengaged_participants = participant_label[participant_label == 0].index.values.copy()

rng = np.random.default_rng(SEED)
rng.shuffle(engaged_participants)
rng.shuffle(disengaged_participants)

def stratified_split(arr, train_frac=0.70, val_frac=0.15):
    n       = len(arr)
    n_train = int(n * train_frac)
    n_val   = int(n * val_frac)
    return arr[:n_train], arr[n_train:n_train + n_val], arr[n_train + n_val:]

train_eng, val_eng, test_eng = stratified_split(engaged_participants)
train_dis, val_dis, test_dis = stratified_split(disengaged_participants)

train_participants = np.concatenate([train_eng, train_dis])
val_participants   = np.concatenate([val_eng,   val_dis])
test_participants  = np.concatenate([test_eng,  test_dis])

train_mask = final_df["participant_id"].isin(train_participants)

clip_meta = (
    final_df.groupby("clip_id")
    .agg(
        engagement         = ("engagement",         "first"),
        participant_id     = ("participant_id",     "first"),
        behavioral_cluster = ("behavioral_cluster", "first")
    )
    .reset_index()
)

train_clips = clip_meta[clip_meta["participant_id"].isin(train_participants)].reset_index(drop=True)
val_clips   = clip_meta[clip_meta["participant_id"].isin(val_participants)].reset_index(drop=True)
test_clips  = clip_meta[clip_meta["participant_id"].isin(test_participants)].reset_index(drop=True)

# Fit scaler on train only
scaler = StandardScaler()
scaler.fit(final_df.loc[train_mask, FEATURE_COLS])
final_df[FEATURE_COLS] = scaler.transform(final_df[FEATURE_COLS])

print(f"  Participants — train: {len(train_participants)}  "
      f"val: {len(val_participants)}  test: {len(test_participants)}")
print(f"  Clips — train: {len(train_clips):,}  "
      f"val: {len(val_clips):,}  test: {len(test_clips):,}")

Loading data...
  Participants — train: 71  val: 14  test: 17
  Clips — train: 8,484  val: 1,832  test: 1,777


In [5]:
# compute cluster-level loss weights for reweighting strategy

cluster_counts  = train_clips["behavioral_cluster"].value_counts().sort_index()
cluster_weights = 1.0 / cluster_counts
cluster_weights = cluster_weights / cluster_weights.sum() * len(cluster_weights)

print("Cluster counts and loss weights (training set):")
for cid in cluster_counts.index:
    print(f"  Cluster {cid}: {cluster_counts[cid]:4d} clips  "
          f"→ weight {cluster_weights[cid]:.4f}")

train_clip_weights = train_clips["behavioral_cluster"].map(cluster_weights).values

Cluster counts and loss weights (training set):
  Cluster 0: 2172 clips  → weight 1.4880
  Cluster 1: 6312 clips  → weight 0.5120


In [6]:
# dataset and dataloaders
class EngagementDataset(Dataset):
    def __init__(self, clips_df, full_df, max_len=MAX_SEQ_LEN):
        self.max_len    = max_len
        self.clip_ids   = clips_df["clip_id"].tolist()
        self.labels     = clips_df["engagement"].tolist()
        self.clusters   = clips_df["behavioral_cluster"].tolist()
        relevant        = full_df[full_df["clip_id"].isin(self.clip_ids)]
        self.clip_dict  = {
            cid: grp.sort_values("timestep")[FEATURE_COLS].values.astype(np.float32)
            for cid, grp in relevant.groupby("clip_id")
        }

    def __len__(self):
        return len(self.clip_ids)

    def __getitem__(self, idx):
        seq = self.clip_dict.get(
            self.clip_ids[idx],
            np.zeros((1, N_FEATURES), dtype=np.float32)
        )
        T = seq.shape[0]
        if T >= self.max_len:
            seq = seq[:self.max_len]
        else:
            seq = np.vstack([seq, np.zeros((self.max_len - T, N_FEATURES),
                                           dtype=np.float32)])
        return (torch.tensor(seq),
                torch.tensor(self.labels[idx],   dtype=torch.long),
                torch.tensor(self.clusters[idx], dtype=torch.long))


print("Building datasets...")
train_dataset = EngagementDataset(train_clips, final_df)
val_dataset   = EngagementDataset(val_clips,   final_df)
test_dataset  = EngagementDataset(test_clips,  final_df)

val_loader  = DataLoader(val_dataset,  batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0)

sampler_weights = torch.tensor(train_clip_weights, dtype=torch.float32)
weighted_sampler = torch.utils.data.WeightedRandomSampler(
    weights     = sampler_weights,
    num_samples = len(sampler_weights),
    replacement = True
)
train_loader_weighted = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                                   sampler=weighted_sampler, num_workers=0)

n_train_pos   = int(train_clips["engagement"].sum())
n_train_neg   = len(train_clips) - n_train_pos
class_weights_tensor = torch.tensor(
    [n_train_pos / n_train_neg, 1.0], dtype=torch.float32
).to(DEVICE)

print("Done.")

Building datasets...
Done.


In [7]:
# model arch and evaluation helpers
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_SEQ_LEN, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe           = torch.zeros(max_len, d_model)
        position     = torch.arange(0, max_len).unsqueeze(1).float()
        div_term     = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


class TransformerClassifier(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, dropout=0.1, n_classes=2,
                 max_len=MAX_SEQ_LEN):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc    = PositionalEncoding(d_model, max_len=max_len,
                                             dropout=dropout)
        encoder_layer   = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.encoder    = nn.TransformerEncoder(encoder_layer,
                                                num_layers=num_layers)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, x, return_representations=False):
        x    = self.pos_enc(self.input_proj(x))
        x    = self.encoder(x)
        rep  = self.dropout(x.mean(dim=1))   # mean-pooled representation
        logits = self.classifier(rep)
        if return_representations:
            return logits, rep
        return logits


def load_best_params():
    with open(BEST_PARAMS) as f:
        p = json.load(f)
    return p

def build_transformer(params):
    return TransformerClassifier(
        input_dim       = N_FEATURES,
        d_model         = int(params["d_model"]),
        nhead           = int(params["nhead"]),
        num_layers      = int(params["num_layers"]),
        dim_feedforward = int(params["dim_feedforward"]),
        dropout         = params["dropout"]
    ).to(DEVICE)


def compute_metrics(y_true, y_pred, split_name=""):
    acc      = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro",  zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro",  zero_division=0)
    print(f"\n── {split_name} ───────────────────────────────────────")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Macro F1:  {macro_f1:.4f}   (primary metric)")
    print(f"  Micro F1:  {micro_f1:.4f}")
    print(classification_report(y_true, y_pred,
                                target_names=["Disengaged", "Engaged"],
                                zero_division=0))
    return {"accuracy": acc, "macro_f1": macro_f1, "micro_f1": micro_f1}


def fairness_metrics(y_true, y_pred, cluster_ids, split_name="", n_bootstrap=1000):
    MEANINGFUL   = 0.10
    cluster_ids  = np.array(cluster_ids)
    unique_clust = sorted(np.unique(cluster_ids))

    def per_cluster_stats(yt, yp, cids):
        rows = []
        for cid in unique_clust:
            mask = cids == cid
            if mask.sum() == 0:
                continue
            yt_c, yp_c = yt[mask], yp[mask]
            n  = len(yt_c)
            tp = int(((yp_c == 1) & (yt_c == 1)).sum())
            tn = int(((yp_c == 0) & (yt_c == 0)).sum())
            fp = int(((yp_c == 1) & (yt_c == 0)).sum())
            fn = int(((yp_c == 0) & (yt_c == 1)).sum())
            rows.append({
                "cluster":        cid,
                "n_clips":        n,
                "tpr":            tp / (tp + fn) if (tp + fn) > 0 else np.nan,
                "fpr":            fp / (fp + tn) if (fp + tn) > 0 else np.nan,
                "fnr":            fn / (tp + fn) if (tp + fn) > 0 else np.nan,
                "error_rate":     (fp + fn) / n,
                "low_confidence": n < 5
            })
        return pd.DataFrame(rows)

    def disparities(df):
        r = df[~df["low_confidence"]]
        return {
            "eod":               r["tpr"].max()        - r["tpr"].min(),
            "fpr_gap":           r["fpr"].max()        - r["fpr"].min(),
            "fnr_gap":           r["fnr"].max()        - r["fnr"].min(),
            "worst_group_error": r["error_rate"].max()
        }

    df    = per_cluster_stats(y_true, y_pred, cluster_ids)
    point = disparities(df)

    boot  = {"eod": [], "fpr_gap": [], "fnr_gap": [], "worst_group_error": []}
    rng_b = np.random.default_rng(SEED)
    n     = len(y_true)
    for _ in range(n_bootstrap):
        idx     = rng_b.integers(0, n, size=n)
        bdf     = per_cluster_stats(y_true[idx], y_pred[idx], cluster_ids[idx])
        brel    = bdf[~bdf["low_confidence"]]
        if len(brel["cluster"].unique()) < 2:
            continue
        b = disparities(brel)
        for k in boot:
            boot[k].append(b[k])

    ci = {k: (np.percentile(v, 2.5), np.percentile(v, 97.5))
          for k, v in boot.items()}

    print(f"\n── Fairness Metrics: {split_name} ───────────────────────")
    for label, key in [
        ("EOD             ", "eod"),
        ("FPR gap         ", "fpr_gap"),
        ("FNR gap         ", "fnr_gap"),
        ("Worst-group err ", "worst_group_error"),
    ]:
        flag = "  ⚠" if point[key] > MEANINGFUL else "  ✓"
        print(f"  {label}: {point[key]:.4f}  "
              f"95% CI [{ci[key][0]:.4f}, {ci[key][1]:.4f}]{flag}")

    if df["low_confidence"].any():
        print(f"  ⚠ Low-confidence clusters: "
              f"{df.loc[df['low_confidence'], 'cluster'].tolist()}")
    print(f"\n  Per-cluster breakdown:")
    print(df.round(4).to_string(index=False))

    return df, {**point, "ci": ci}


def run_inference(model, loader):
    model.eval()
    preds, labels, clusters = [], [], []
    with torch.no_grad():
        for X_batch, y_batch, c_batch in loader:
            preds.extend(
                model(X_batch.to(DEVICE)).argmax(dim=1).cpu().numpy()
            )
            labels.extend(y_batch.numpy())
            clusters.extend(c_batch.numpy())
    return np.array(labels), np.array(preds), np.array(clusters)

In [8]:
# Cell 6 — MMD loss for feature regularisation
def mmd_loss(representations, cluster_ids, unique_clusters):
    cluster_means = []
    for cid in unique_clusters:
        mask = cluster_ids == cid
        if mask.sum() < 2:
            return torch.tensor(0.0, device=representations.device)
        cluster_means.append(representations[mask].mean(dim=0))

    mmd = torch.tensor(0.0, device=representations.device)
    n_pairs = 0
    for i in range(len(cluster_means)):
        for j in range(i + 1, len(cluster_means)):
            diff  = cluster_means[i] - cluster_means[j]
            mmd  += (diff * diff).sum()
            n_pairs += 1

    return mmd / n_pairs if n_pairs > 0 else mmd

UNIQUE_TRAIN_CLUSTERS = sorted(train_clips["behavioral_cluster"].unique().tolist())
print(f"Training clusters: {UNIQUE_TRAIN_CLUSTERS}")

Training clusters: [0, 1]


In [9]:
# Cell 7 — Fair model training function
def train_fair_model(model, lambda_reweight, lambda_mmd, learning_rate,
                     use_weighted_sampler=True):

    loader = train_loader_weighted if use_weighted_sampler else DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    )

    criterion = nn.CrossEntropyLoss(
        weight    = class_weights_tensor,
        reduction = "none"
    )
    optimiser         = torch.optim.Adam(model.parameters(), lr=learning_rate)
    best_val_f1       = -1.0
    best_state        = None
    epochs_no_improve = 0

    cw_map = torch.tensor(
        [cluster_weights.get(cid, 1.0) for cid in UNIQUE_TRAIN_CLUSTERS],
        dtype=torch.float32
    ).to(DEVICE)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0

        for X_batch, y_batch, c_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            c_batch = c_batch.to(DEVICE)

            optimiser.zero_grad()
            logits, reps = model(X_batch, return_representations=True)

            ce_per_sample = criterion(logits, y_batch)   # (batch,)
            if lambda_reweight > 0:
                # Scale each sample's loss by its cluster's weight
                sample_weights = torch.stack([
                    cw_map[UNIQUE_TRAIN_CLUSTERS.index(int(c))]
                    for c in c_batch
                ])
                ce_loss = (ce_per_sample * sample_weights).mean()
            else:
                ce_loss = ce_per_sample.mean()

            mmd = mmd_loss(reps, c_batch, UNIQUE_TRAIN_CLUSTERS) \
                  if lambda_mmd > 0 else torch.tensor(0.0)

            loss = ce_loss + lambda_reweight * ce_loss + lambda_mmd * mmd
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()
            epoch_loss += loss.item()

        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for X_batch, y_batch, _ in val_loader:
                val_preds.extend(
                    model(X_batch.to(DEVICE)).argmax(dim=1).cpu().numpy()
                )
                val_labels.extend(y_batch.numpy())

        val_f1 = f1_score(val_labels, val_preds, average="macro", zero_division=0)

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d} | loss: {epoch_loss/len(loader):.4f} | "
                  f"val macro F1: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1       = val_f1
            best_state        = {k: v.cpu().clone()
                                 for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"  Early stopping at epoch {epoch} "
                      f"(best val macro F1: {best_val_f1:.4f})")
                break

    model.load_state_dict(best_state)
    return best_val_f1, model

In [10]:
# lambda grid search

params = load_best_params()

LAMBDA_GRID = {
    "lambda_reweight": [0.0, 0.01, 0.1, 0.5, 1.0],
    "lambda_mmd":      [0.0, 0.01, 0.1, 0.5, 1.0],
}

from itertools import product as iproduct
lambda_combos = [
    (lr, lm)
    for lr, lm in iproduct(LAMBDA_GRID["lambda_reweight"],
                            LAMBDA_GRID["lambda_mmd"])
    if not (lr == 0.0 and lm == 0.0)
]
lambda_combos = [(0.0, 0.0)] + lambda_combos 
print(f"Lambda configurations to search: {len(lambda_combos)}")

lambda_results = []

for i, (lw, lm) in enumerate(lambda_combos):
    print(f"\n[{i+1}/{len(lambda_combos)}] "
          f"lambda_reweight={lw}  lambda_mmd={lm}")
    model = build_transformer(params)
    val_f1, trained = train_fair_model(
        model,
        lambda_reweight      = lw,
        lambda_mmd           = lm,
        learning_rate        = params["learning_rate"],
        use_weighted_sampler = (lw > 0)
    )

    val_labels, val_preds, val_clusters = run_inference(trained, val_loader)
    val_clusters = val_clips["behavioral_cluster"].values.astype(int)
    reliable_mask = pd.Series(val_clusters).isin(
        val_clips["behavioral_cluster"].value_counts()[
            val_clips["behavioral_cluster"].value_counts() >= 5
        ].index
    ).values
    if reliable_mask.sum() > 0:
        val_worst = max(
            (((val_preds[val_clusters == cid] !=
               val_labels[val_clusters == cid]).sum() /
              (val_clusters == cid).sum())
             for cid in np.unique(val_clusters)
             if (val_clusters == cid).sum() >= 5),
            default=np.nan
        )
    else:
        val_worst = np.nan

    lambda_results.append({
        "lambda_reweight": lw,
        "lambda_mmd":      lm,
        "val_macro_f1":    val_f1,
        "val_worst_group": val_worst,
    })
    print(f"  val macro F1: {val_f1:.4f}  "
          f"val worst-group error: {val_worst:.4f}")

lambda_df = pd.DataFrame(lambda_results).sort_values(
    "val_macro_f1", ascending=False
)
print("\n── Lambda search results (sorted by val macro F1) ──────")
print(lambda_df.round(4).to_string(index=False))

Lambda configurations to search: 25

[1/25] lambda_reweight=0.0  lambda_mmd=0.0
  Epoch   1 | loss: 0.9995 | val macro F1: 0.6209
  Epoch  10 | loss: 0.8203 | val macro F1: 0.6720
  Epoch  20 | loss: 0.8123 | val macro F1: 0.6151
  Early stopping at epoch 20 (best val macro F1: 0.6720)
  val macro F1: 0.6720  val worst-group error: 0.3107

[2/25] lambda_reweight=0.0  lambda_mmd=0.01
  Epoch   1 | loss: 1.0188 | val macro F1: 0.6234
  Epoch  10 | loss: 0.8206 | val macro F1: 0.5869
  Early stopping at epoch 13 (best val macro F1: 0.6606)
  val macro F1: 0.6606  val worst-group error: 0.2961

[3/25] lambda_reweight=0.0  lambda_mmd=0.1
  Epoch   1 | loss: 1.1419 | val macro F1: 0.5947
  Epoch  10 | loss: 0.8597 | val macro F1: 0.6567
  Early stopping at epoch 12 (best val macro F1: 0.6779)
  val macro F1: 0.6779  val worst-group error: 0.2512

[4/25] lambda_reweight=0.0  lambda_mmd=0.5
  Epoch   1 | loss: 1.5022 | val macro F1: 0.5724
  Epoch  10 | loss: 0.8654 | val macro F1: 0.5642
  Ea

In [11]:
# select best lambda and retrain final fair model
baseline_row     = lambda_df[
    (lambda_df["lambda_reweight"] == 0.0) &
    (lambda_df["lambda_mmd"]      == 0.0)
].iloc[0]
baseline_val_worst = baseline_row["val_worst_group"]

improved = lambda_df[
    (lambda_df["val_worst_group"] < baseline_val_worst) &
    ~((lambda_df["lambda_reweight"] == 0.0) &
      (lambda_df["lambda_mmd"]      == 0.0))
]

if len(improved) == 0:
    print("No config improved worst-group error on validation set.")
    print("Falling back to config with best val macro F1.")
    best_row = lambda_df.iloc[0]
else:
    best_row = improved.sort_values("val_macro_f1", ascending=False).iloc[0]

best_lw = best_row["lambda_reweight"]
best_lm = best_row["lambda_mmd"]
print(f"Selected: lambda_reweight={best_lw}  lambda_mmd={best_lm}")
print(f"  val macro F1:    {best_row['val_macro_f1']:.4f}")
print(f"  val worst-group: {best_row['val_worst_group']:.4f}  "
      f"(baseline: {baseline_val_worst:.4f})")

print("\nRetraining final fair model from scratch...")
fair_model = build_transformer(params)
_, fair_model = train_fair_model(
    fair_model,
    lambda_reweight      = best_lw,
    lambda_mmd           = best_lm,
    learning_rate        = params["learning_rate"],
    use_weighted_sampler = (best_lw > 0)
)
torch.save(fair_model.state_dict(), "transformer_fair.pt")
print("Saved: transformer_fair.pt")

Selected: lambda_reweight=0.5  lambda_mmd=0.1
  val macro F1:    0.6810
  val worst-group: 0.2937  (baseline: 0.3107)

Retraining final fair model from scratch...
  Epoch   1 | loss: 1.6267 | val macro F1: 0.6219
  Epoch  10 | loss: 1.1899 | val macro F1: 0.5851
  Epoch  20 | loss: 1.0914 | val macro F1: 0.6344
  Early stopping at epoch 24 (best val macro F1: 0.6701)
Saved: transformer_fair.pt


In [12]:
# full test-set evaluation of fair model
print("=" * 60)
print("FAIR MODEL — TEST SET EVALUATION")
print("=" * 60)

y_true, y_pred, cluster_ids = run_inference(fair_model, test_loader)
cluster_ids = test_clips["behavioral_cluster"].values.astype(int)

fair_metrics  = compute_metrics(y_true, y_pred, split_name="Fair Transformer — Test")
_, fair_disp  = fairness_metrics(y_true, y_pred, cluster_ids,
                                  split_name="Fair Transformer")

FAIR MODEL — TEST SET EVALUATION

── Fair Transformer — Test ───────────────────────────────────────
  Accuracy:  0.7817
  Macro F1:  0.6248   (primary metric)
  Micro F1:  0.7817
              precision    recall  f1-score   support

  Disengaged       0.44      0.34      0.38       356
     Engaged       0.84      0.89      0.87      1421

    accuracy                           0.78      1777
   macro avg       0.64      0.62      0.62      1777
weighted avg       0.76      0.78      0.77      1777


── Fairness Metrics: Fair Transformer ───────────────────────
  EOD             : 0.1265  95% CI [0.0829, 0.1710]  ⚠
  FPR gap         : 0.4166  95% CI [0.3116, 0.5044]  ⚠
  FNR gap         : 0.1265  95% CI [0.0829, 0.1710]  ⚠
  Worst-group err : 0.2442  95% CI [0.2127, 0.2813]  ⚠

  Per-cluster breakdown:
 cluster  n_clips    tpr    fpr    fnr  error_rate  low_confidence
       0      516 0.8015 0.3902 0.1985      0.2442           False
       1     1261 0.9280 0.8069 0.0720      0.2078

In [13]:
# compare against baseline transformer
print("=" * 60)
print("COMPARISON: BASELINE vs FAIR TRANSFORMER (test set)")
print("=" * 60)

baseline_model = build_transformer(params)
baseline_model.load_state_dict(
    torch.load(BASELINE_PT, map_location=DEVICE)
)
y_true_b, y_pred_b, _ = run_inference(baseline_model, test_loader)
cluster_ids_b          = test_clips["behavioral_cluster"].values.astype(int)
base_metrics           = compute_metrics(y_true_b, y_pred_b,
                             split_name="Baseline Transformer — Test")
_, base_disp           = fairness_metrics(y_true_b, y_pred_b, cluster_ids_b,
                             split_name="Baseline Transformer")

print("\n── Overall Performance ─────────────────────────────────")
perf_df = pd.DataFrame({
    "Model":    ["Baseline Transformer", "Fair Transformer"],
    "Accuracy": [base_metrics["accuracy"],  fair_metrics["accuracy"]],
    "Macro F1": [base_metrics["macro_f1"],  fair_metrics["macro_f1"]],
    "Micro F1": [base_metrics["micro_f1"],  fair_metrics["micro_f1"]],
})
print(perf_df.round(4).to_string(index=False))

print("\n── Fairness Disparities ────────────────────────────────")
MEANINGFUL = 0.10

def fmt(val, ci_lo, ci_hi, threshold=MEANINGFUL):
    flag = " ⚠" if val > threshold else "  "
    return f"{val:.4f}  [{ci_lo:.4f}, {ci_hi:.4f}]{flag}"

rows = []
for name, disp in [("Baseline", base_disp), ("Fair", fair_disp)]:
    ci = disp["ci"]
    rows.append({
        "Model":             name,
        "EOD":               fmt(disp["eod"],               *ci["eod"]),
        "FPR gap":           fmt(disp["fpr_gap"],           *ci["fpr_gap"]),
        "FNR gap":           fmt(disp["fnr_gap"],           *ci["fnr_gap"]),
        "Worst-group error": fmt(disp["worst_group_error"], *ci["worst_group_error"],
                                 threshold=0.0),
    })
print(pd.DataFrame(rows).to_string(index=False))

print("\n── Effectiveness Check (Section 3.5.3) ─────────────────")
checks = {
    f"Macro F1 ≥ {MACRO_F1_FLOOR:.4f}":
        fair_metrics["macro_f1"] >= MACRO_F1_FLOOR,
    f"Worst-group error < {BASELINE_WORST_GROUP_ERR:.4f}":
        fair_disp["worst_group_error"] < BASELINE_WORST_GROUP_ERR,
    f"EOD < {BASELINE_EOD:.4f}":
        fair_disp["eod"] < BASELINE_EOD,
}
all_passed = True
for criterion, passed in checks.items():
    status = "✓ PASSED" if passed else "✗ FAILED"
    print(f"  {status}  {criterion}")
    if not passed:
        all_passed = False

print(f"\n  {'✓ Fair model meets all criteria.' if all_passed else '✗ Not all criteria met — consider adjusting lambdas.'}")

# save lambda config used
with open("fair_model_config.json", "w") as f:
    json.dump({
        "lambda_reweight": best_lw,
        "lambda_mmd":      best_lm,
        "val_macro_f1":    float(best_row["val_macro_f1"]),
        "val_worst_group": float(best_row["val_worst_group"]),
    }, f, indent=2)
print("\nSaved: fair_model_config.json")

COMPARISON: BASELINE vs FAIR TRANSFORMER (test set)

── Baseline Transformer — Test ───────────────────────────────────────
  Accuracy:  0.7203
  Macro F1:  0.6142   (primary metric)
  Micro F1:  0.7203
              precision    recall  f1-score   support

  Disengaged       0.36      0.49      0.41       356
     Engaged       0.86      0.78      0.82      1421

    accuracy                           0.72      1777
   macro avg       0.61      0.63      0.61      1777
weighted avg       0.76      0.72      0.74      1777


── Fairness Metrics: Baseline Transformer ───────────────────────
  EOD             : 0.1508  95% CI [0.0958, 0.2073]  ⚠
  FPR gap         : 0.4581  95% CI [0.3584, 0.5507]  ⚠
  FNR gap         : 0.1508  95% CI [0.0958, 0.2073]  ⚠
  Worst-group err : 0.3023  95% CI [0.2707, 0.3430]  ⚠

  Per-cluster breakdown:
 cluster  n_clips    tpr    fpr    fnr  error_rate  low_confidence
       0      516 0.6692 0.2114 0.3308      0.3023           False
       1     1261 0.820